### Getting started with LangChain, LangSmith and LangServe

In [32]:
# load environment variables
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

#Langchain Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true" #Enables langchain tracing
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT") #Check on the portal to track the traces of requests  

In [33]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-6-luna")
print(llm)

metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-openai': '1.8.0'}} profile={'name': 'GPT-6 Luna', 'release_date': '2026-09-22', 'last_updated': '2026-09-22', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium', 'mid_conversation_system_messages': True, 'mid_conversation_tools': False} client=<openai.resources.chat.completions.completions.Completions object at 0x12461

In [34]:
results = llm.invoke("Why did Learjet crash?")

In [35]:
print(results)

content='Which Learjet crash do you mean? If you mean the **2023 crash in Virginia**, investigators concluded the pilot likely became incapacitated by **hypoxia** after the cabin lost pressurization; the aircraft continued flying until it ran out of fuel and crashed.\n\nIf you mean another crash—such as the **2025 Philadelphia Learjet crash**—tell me which one.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 597, 'prompt_tokens': 12, 'total_tokens': 609, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Ek5GmvWxas8LtNaA2LVI79ixIdh8t', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': Non

In [36]:
#Prompt Template = Our prompt + The query
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert AI engineer. Provide me answers based on the question"),
        ("user", "{input}")
    ]
)

In [37]:
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI engineer. Provide me answers based on the question'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [38]:
#Method chaining
chain = prompt|llm
response = chain.invoke({"input":"Can you tell me about NLP?"})
print(response)

content='**Natural Language Processing (NLP)** is a branch of artificial intelligence that helps computers work with human language—such as text and speech. It combines ideas from computer science, machine learning, and linguistics.\n\nNLP systems can be designed to **analyze language**, **generate it**, or do both. Common tasks include:\n\n- **Translation:** converting text from one language to another  \n- **Sentiment analysis:** identifying opinions or emotional tone  \n- **Information extraction:** finding names, dates, places, or key facts in text  \n- **Speech recognition:** turning spoken words into text  \n- **Text generation:** drafting answers, summaries, or other content  \n- **Question answering and chatbots:** responding to questions in natural language  \n\nOlder NLP systems often relied on hand-written rules or statistical methods. Many modern systems use **machine learning**, especially **transformer-based neural networks**. Large language models are one kind of these s

In [39]:
type(response)

langchain_core.messages.ai.AIMessage

In [40]:
## strOutput Parser to get the text

from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
chain = prompt|llm|output_parser

response = chain.invoke({"input":"Can you tell me about Langsmith"})
print(response)

**LangSmith** is LangChain’s platform for building, testing, and monitoring applications that use large language models. It’s not an LLM itself; it helps you understand and improve the app that calls one.

Key features include:

- **Tracing:** Records the steps in an LLM workflow—such as model calls, prompts, tool use, inputs, outputs, latency, and errors.
- **Evaluation:** Lets you test application outputs against datasets and use automated or human feedback to compare changes.
- **Prompt management:** Helps organize and version prompts so you can test updates and track which version an app used.
- **Production monitoring:** Helps spot failures, performance issues, and changes in output quality.

It works especially well with **LangChain** and **LangGraph**, but can also be used with other LLM frameworks and custom applications.

A typical workflow is to trace an app during development, evaluate it on a set of examples, then monitor traces from real usage. Because traces can contain p